In [10]:
import scanpy as sc
import jax
import os
from cellflow.metrics import compute_metrics, compute_mean_metrics, compute_metrics_fast, compute_e_distance, compute_r_squared
import cellflow.preprocessing as cfpp
import anndata as ad
import pandas as pd
import numpy as np
import sys
import pickle

In [2]:
def get_embedding(embedding: str) -> dict[str, np.ndarray]:
    if embedding == "esm2":
        adata_tmp = sc.read_h5ad("/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata/adata_train_pca_50_split_0.h5ad")
        return adata_tmp.uns["esm2"]
    if embedding == "nargab":
        with open('/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/gene_nargab.pkl', 'rb') as f:
            nargab_emb = pickle.load(f)
            return nargab_emb
    else:
        raise ValueError("Embedding not supported")


In [29]:
DATA_DIR = "/lustre/groups/ml01/workspace/ot_perturbation/data/norman_new"
split, embedding_str = "0", "esm2"  #sys.argv[1], sys.argv[2]

adata_train_path = os.path.join(DATA_DIR, f"adata_train_pca_50_split_{split}.h5ad")
adata_test_path = os.path.join(DATA_DIR, f"adata_test_pca_50_split_{split}.h5ad")
adata_ood_path = os.path.join(DATA_DIR, f"adata_ood_pca_50_split_{split}.h5ad")

# load data splits
adata_train = sc.read(adata_train_path)
adata_test = sc.read(adata_test_path)
adata_ood = sc.read(adata_ood_path)

In [30]:
conds = adata_ood.obs.drop_duplicates()

In [31]:
conds

,condition,cell_type,dose_val,control,condition_name,cell_line,gene_1,gene_2,num_control,kategory,subgroup
cell_barcode,,,,,,,,,,,
AAACCTGGTATAATGG-1,ctrl,A549,1,True,A549_ctrl_1,A549,ctrl,ctrl,2,ctrl,single
AAACCTGGTATCGCAT-1,CBL+PTPN9,A549,1+1,False,A549_CBL+PTPN9_1+1,A549,CBL,PTPN9,0,double,double_seen_0
AAACCTGGTCTGATTG-1,DUSP9+ctrl,A549,1+1,False,A549_DUSP9+ctrl_1+1,A549,DUSP9,ctrl,1,single,single
AAACCTGGTTCACCTC-1,MAP2K6+SPI1,A549,1+1,False,A549_MAP2K6+SPI1_1+1,A549,MAP2K6,SPI1,0,double,double_seen_1
AAACCTGTCAGCGATT-1,UBASH3B+PTPN12,A549,1+1,False,A549_UBASH3B+PTPN12_1+1,A549,UBASH3B,PTPN12,0,double,double_seen_2
...,...,...,...,...,...,...,...,...,...,...,...
AGTGAGGTCCTCAACC-1,IGDCC3+PRTG,A549,1+1,False,A549_IGDCC3+PRTG_1+1,A549,IGDCC3,PRTG,0,double,double_seen_1
ATAGACCGTAGCAAAT-1,CDKN1C+ctrl,A549,1+1,False,A549_CDKN1C+ctrl_1+1,A549,CDKN1C,ctrl,1,single,single
ATTGGACTCCCAAGAT-1,ctrl+CDKN1B,A549,1+1,False,A549_ctrl+CDKN1B_1+1,A549,ctrl,CDKN1B,1,single,single


In [32]:

from typing import Tuple, Any




def get_mask(x, y):
    return x[:, [gene in y for gene in adata_train.var_names]]

def get_train_embeddings(adata_train: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_train.obs.drop_duplicates()
    emb_vectors = {}
    for _,row in conds.iterrows():
        gene_1 = row["gene_1"]
        gene_2  = row["gene_2"]
        if gene_1=="ctrl" and gene_2=="ctrl":
            continue
        elif gene_1 != "ctrl" and gene_2!= "ctrl":
            emb_vectors[(gene_1, gene_2)] = (embeddings[gene_1] + embeddings[gene_2])/2.0
        elif gene_1 != "ctrl" and gene_2=="ctrl":
            emb_vectors[(gene_1, gene_2)] = embeddings[gene_1]
        elif gene_1=="ctrl" and gene_2!="ctrl":
            emb_vectors[(gene_1, gene_2)] = embeddings[gene_2]
    return emb_vectors
        

def find_closest_emb(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb


embeddings = get_embedding(embedding_str)
train_embeddings = get_train_embeddings(adata_train, embeddings)
conds = adata_ood.obs.drop_duplicates()
preds = {}

for _,row in conds.iterrows():    
    if row["condition"] == "ctrl":
        continue
    elif row["gene_2"] =="ctrl":
        emb_0 = row["gene_1"]
        closest_emb = find_closest_emb(embeddings[emb_0], train_embeddings)
    elif row["gene_1"] =="ctrl":
        emb_0 = row["gene_2"]
        closest_emb = find_closest_emb(embeddings[emb_0], train_embeddings)
    else:
        emb_0 = (embeddings[row["gene_1"]] + embeddings[row["gene_2"]])/2.0
        closest_emb = find_closest_emb(emb_0, train_embeddings)
    gene1, gene2 = row["gene_1"], row["gene_2"]
    print(closest_emb)
    preds[row["condition"]] = adata_train[adata_train.obs["condition"]==f"{closest_emb[0]}+{closest_emb[1]}"].X.toarray()
        

('SAMD1', 'TGFBR2')
('SAMD1', 'UBASH3B')
('MAP4K5', 'ctrl')
('PTPN12', 'UBASH3A')
('KMT2A', 'ctrl')
('AHR', 'FEV')
('IRF1', 'SET')
('FOXA3', 'FOXA1')
('TBX3', 'ctrl')
('SLC6A9', 'ctrl')
('IGDCC3', 'MAPK1')
('COL1A1', 'ctrl')
('ctrl', 'CEBPE')
('SNAI1', 'UBASH3B')
('UBASH3B', 'UBASH3A')
('SNAI1', 'UBASH3B')
('FEV', 'MAP7D1')
('UBASH3B', 'ctrl')
('IGDCC3', 'MAPK1')
('TGFBR2', 'IGDCC3')
('CEBPE', 'RUNX1T1')
('SAMD1', 'UBASH3B')
('AHR', 'FEV')
('IGDCC3', 'MAPK1')
('MAP2K6', 'IKZF3')
('CEBPE', 'RUNX1T1')
('PTPN12', 'SNAI1')
('CEBPE', 'PTPN12')
('POU3F2', 'ctrl')
('ETS2', 'IGDCC3')
('PTPN12', 'SNAI1')
('TGFBR2', 'IGDCC3')
('AHR', 'FEV')
('ctrl', 'CEBPA')
('ETS2', 'IGDCC3')
('ctrl', 'ETS2')
('LYL1', 'ctrl')
('MAPK1', 'IKZF3')
('FOXA1', 'FOXF1')
('FOXA3', 'FOXA1')
('CEBPE', 'RUNX1T1')
('PTPN12', 'UBASH3A')
('CEBPE', 'RUNX1T1')
('SAMD1', 'UBASH3B')
('BAK1', 'ctrl')
('FOXF1', 'HOXB9')
('TSC22D1', 'ctrl')
('COL1A1', 'ctrl')
('ZC3HAV1', 'CEBPE')
('ZC3HAV1', 'CEBPE')
('UBASH3B', 'CNN1')
('PTPN12', 

In [33]:
import anndata as ad
import pandas as pd
all_data = []
conditions = []

for condition, array in preds.items():
    
    all_data.append(array)
    conditions.extend([condition] * array.shape[0])

# Stack all data vertically to create a single array
all_data_array = np.vstack(all_data)

# Create a DataFrame for the .obs attribute
obs_data = pd.DataFrame({
    'condition': conditions
})

# Create the Anndata object
adata_pred_ood = ad.AnnData(X=all_data_array, obs=obs_data)

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [103]:
adata_ref = ad.concat((adata_train, adata_test, adata_ood[adata_ood.obs["control"]==0]))
cfpp.centered_pca(adata_ref, n_comps=10)


cfpp.project_pca(query_adata=adata_pred_ood, ref_adata=adata_ref)
cfpp.project_pca(query_adata=adata_ood, ref_adata=adata_ref)
ood_data_target_encoded = {}
ood_data_target_decoded = {}
ood_data_target_encoded_predicted = {}
ood_data_target_decoded_predicted = {}
for cond in adata_ood.obs["condition"].cat.categories:
    if cond == "ctrl":
        continue
    ood_data_target_encoded[cond] = adata_ood[adata_ood.obs["condition"] == cond].obsm["X_pca"]
    ood_data_target_decoded[cond] = adata_ood[adata_ood.obs["condition"] == cond].X.toarray()
    ood_data_target_decoded_predicted[cond] = adata_pred_ood[adata_pred_ood.obs["condition"] == cond].X.toarray()
    ood_data_target_encoded_predicted[cond] = adata_pred_ood[adata_pred_ood.obs["condition"] == cond].obsm["X_pca"]


ood_deg_dict = {
    k: v
    for k, v in adata_train.uns["rank_genes_groups_cov_all"].items()
    if k in ood_data_target_decoded_predicted.keys()
}


def get_mask(x, y):
    return x[:, [gene in y for gene in adata_train.var_names]]


ood_deg_target_decoded_predicted = jax.tree_util.tree_map(get_mask, ood_data_target_decoded_predicted, ood_deg_dict)
ood_deg_target_decoded = jax.tree_util.tree_map(get_mask, ood_data_target_decoded, ood_deg_dict)


ood_metrics_encoded = jax.tree_util.tree_map(
    compute_metrics_fast, ood_data_target_encoded, ood_data_target_encoded_predicted
)

ood_metrics_decoded = jax.tree_util.tree_map(
    compute_metrics+, ood_data_target_decoded, ood_data_target_decoded_predicted
)

ood_metrics_deg = jax.tree_util.tree_map(
    compute_metrics_fast, ood_deg_target_decoded, ood_deg_target_decoded_predicted
)



In [104]:
single = conds[conds["subgroup"]=="single"]["condition"].values
double_0 = conds[conds["subgroup"]=="double_seen_0"]["condition"].values
double_1 = conds[conds["subgroup"]=="double_seen_1"]["condition"].values
double_2 = conds[conds["subgroup"]=="double_seen_2"]["condition"].values


esm2

In [37]:
e_single = [v for k,v in ood_metrics_encoded.items() if k in single]
np.mean(e_single)

32.78075906965468

In [38]:
e_d0 = [v for k,v in ood_metrics_encoded.items() if k in double_0]
np.mean(e_d0)

42.04621728261312

In [39]:
e_d1 = [v for k,v in ood_metrics_encoded.items() if k in double_1]
np.mean(e_d1)

33.43922467665239

In [40]:
e_d2 = [v for k,v in ood_metrics_encoded.items() if k in double_2]
np.mean(e_d2)

29.76801300048828

In [42]:
r_single = [v for k,v in ood_metrics_decoded.items() if k in single]
np.mean(r_single)

0.9629667599995931

In [43]:
r_d0 = [v for k,v in ood_metrics_decoded.items() if k in double_0]
np.mean(r_d0)

0.9565836985905966

In [44]:
r_d1 = [v for k,v in ood_metrics_decoded.items() if k in double_1]
np.mean(r_d1)

0.9694310440258547

In [45]:
r_d2 = [v for k,v in ood_metrics_decoded.items() if k in double_2]
np.mean(r_d2)

0.973874584833781

In [105]:
df_enc = pd.DataFrame.from_dict(ood_metrics_encoded,orient="index")#.to_csv(os.path.join(config_dict["training"]["out_dir"], f"{wandb.run.name}_{condition}.csv"))
df_dec = pd.DataFrame.from_dict(ood_metrics_decoded,orient="index")
df_deg = pd.DataFrame.from_dict(ood_metrics_deg,orient="index")

In [106]:
df_dec

,r_squared,e_distance,mmd
AHR+KLF1,0.927090,126.558289,0.026198
ARID1A+ctrl,0.965542,52.679321,0.015930
BCL2L11+BAK1,0.997289,4.510925,0.008605
BCL2L11+TGFBR2,0.987476,20.501587,0.010963
BCL2L11+ctrl,0.972371,45.407379,0.015129
...,...,...,...
ctrl+MEIS1,0.919625,133.079895,0.023629
ctrl+OSR2,0.980623,32.554474,0.012662
ctrl+PRTG,0.987989,19.272095,0.009120
ctrl+PTPN9,0.971522,45.594788,0.016892


In [107]:
for col in df_enc.columns:
    df_enc[f"encoded_ood_{col}"] = df_enc[col]
    del df_enc[col]

for col in df_dec.columns:
    df_dec[f"decoded_ood_{col}"] = df_dec[col]
    del df_dec[col]

for col in df_deg.columns:
    df_deg[f"deg_ood_{col}"] = df_deg[col]
    del df_deg[col]


In [108]:
df_enc["condition"] = df_enc.index



In [109]:
df_all = pd.concat((df_enc, df_dec, df_deg), axis=1)

In [110]:
df_all.head()

,encoded_ood_r_squared,encoded_ood_e_distance,encoded_ood_mmd,condition,decoded_ood_r_squared,decoded_ood_e_distance,decoded_ood_mmd,deg_ood_r_squared,deg_ood_e_distance,deg_ood_mmd
AHR+KLF1,-2.908050,74.546616,0.098283,AHR+KLF1,0.927090,126.558289,0.026198,0.736697,29.262005,0.068721
ARID1A+ctrl,-0.856667,26.093658,0.049264,ARID1A+ctrl,0.965542,52.679321,0.015930,0.888779,17.823984,0.056400
BCL2L11+BAK1,0.958315,0.396355,0.006472,BCL2L11+BAK1,0.997289,4.510925,0.008605,0.322746,0.454313,0.070491
BCL2L11+TGFBR2,-0.447892,11.815014,0.027503,BCL2L11+TGFBR2,0.987476,20.501587,0.010963,0.935022,3.875095,0.022408
BCL2L11+ctrl,-3.978754,32.543152,0.054625,BCL2L11+ctrl,0.972371,45.407379,0.015129,0.979700,0.401800,0.036001


In [111]:
df_deg

,deg_ood_r_squared,deg_ood_e_distance,deg_ood_mmd
AHR+KLF1,0.736697,29.262005,0.068721
ARID1A+ctrl,0.888779,17.823984,0.056400
BCL2L11+BAK1,0.322746,0.454313,0.070491
BCL2L11+TGFBR2,0.935022,3.875095,0.022408
BCL2L11+ctrl,0.979700,0.401800,0.036001
...,...,...,...
ctrl+MEIS1,0.863198,20.402870,0.083523
ctrl+OSR2,0.812193,13.373486,0.040092
ctrl+PRTG,0.951605,8.288507,0.025395
ctrl+PTPN9,0.964042,3.501163,0.034854


In [112]:
df_cat = pd.concat((adata_train.obs.drop_duplicates(subset="condition"),adata_ood.obs.drop_duplicates(subset="condition")))
cond_to_cat = df_cat.set_index("condition")["subgroup"].to_dict()

In [113]:
df_all["subgroup"] = df_all["condition"].map(cond_to_cat)
df_all["model"] = "closest_embedding"

In [114]:
df_all.head()

,encoded_ood_r_squared,encoded_ood_e_distance,encoded_ood_mmd,condition,decoded_ood_r_squared,decoded_ood_e_distance,decoded_ood_mmd,deg_ood_r_squared,deg_ood_e_distance,deg_ood_mmd,subgroup,model
AHR+KLF1,-2.908050,74.546616,0.098283,AHR+KLF1,0.927090,126.558289,0.026198,0.736697,29.262005,0.068721,double_seen_1,closest_embedding
ARID1A+ctrl,-0.856667,26.093658,0.049264,ARID1A+ctrl,0.965542,52.679321,0.015930,0.888779,17.823984,0.056400,single,closest_embedding
BCL2L11+BAK1,0.958315,0.396355,0.006472,BCL2L11+BAK1,0.997289,4.510925,0.008605,0.322746,0.454313,0.070491,double_seen_1,closest_embedding
BCL2L11+TGFBR2,-0.447892,11.815014,0.027503,BCL2L11+TGFBR2,0.987476,20.501587,0.010963,0.935022,3.875095,0.022408,double_seen_1,closest_embedding
BCL2L11+ctrl,-3.978754,32.543152,0.054625,BCL2L11+ctrl,0.972371,45.407379,0.015129,0.979700,0.401800,0.036001,single,closest_embedding


In [115]:
df_old = pd.read_csv("/lustre/groups/ml01/workspace/ot_perturbation/data/norman_2/results/allocation/norman_results_all.csv")

In [116]:
ddf = pd.concat((df_old, df_all), axis=0)

In [117]:
ddf.groupby("model")[["decoded_ood_r_squared", "encoded_ood_e_distance"]].mean()

,decoded_ood_r_squared,encoded_ood_e_distance
model,,
additive,0.989467,8.648598
biolord,0.965892,23.422250
cellflow,0.979322,21.237915
closest_embedding,0.966438,33.668299
gears,0.970892,20.350875
identity,0.959597,51.366352
scgpt,0.974246,26.728971


In [118]:
ddf.head()

,Unnamed: 0,subgroup,condition,encoded_ood_r_squared,encoded_ood_sinkhorn_div_1,encoded_ood_sinkhorn_div_10,encoded_ood_sinkhorn_div_100,encoded_ood_e_distance,encoded_ood_mmd,decoded_ood_r_squared,...,deg_ood_sinkhorn_div_1,deg_ood_sinkhorn_div_10,deg_ood_sinkhorn_div_100,deg_ood_e_distance,deg_ood_mmd,seed,model,decoded_ood_sinkhorn_div_1,decoded_ood_sinkhorn_div_10,decoded_ood_sinkhorn_div_100
0,0,double_seen_1,AHR+KLF1,-0.235969,26.363504,20.453796,13.004532,23.576266,0.088512,0.965454,...,49.004730,23.053215,9.580486,18.465538,0.055382,1.0,biolord,NaN,NaN,NaN
1,1,single,ARID1A+ctrl,-0.870820,32.711014,26.043972,15.835747,26.292580,0.075837,0.957185,...,51.148735,24.314388,9.643890,18.312755,0.052491,1.0,biolord,NaN,NaN,NaN
2,2,double_seen_1,BCL2L11+BAK1,0.912641,8.554052,2.861893,0.661819,0.830657,0.019190,0.984661,...,40.841179,14.373302,0.673000,0.470072,0.452304,1.0,biolord,NaN,NaN,NaN
3,3,double_seen_1,BCL2L11+TGFBR2,0.863316,9.772830,4.568264,0.941708,1.115361,0.019349,0.985263,...,38.525169,14.326378,1.011749,1.303958,0.060637,1.0,biolord,NaN,NaN,NaN
4,4,single,BCL2L11+ctrl,0.888504,9.687116,4.609301,0.755690,0.728787,0.016104,0.986481,...,39.652954,13.896275,0.832378,0.833630,0.280454,1.0,biolord,NaN,NaN,NaN


In [125]:
split = 1

adata_train_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/combosciplex/adata_train_{split}.h5ad"
adata_test_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/combosciplex/adata_test_{split}.h5ad"
adata_ood_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/combosciplex/adata_ood_{split}.h5ad"
adata_train = sc.read(adata_train_path)
adata_test = sc.read(adata_test_path)
adata_ood = sc.read(adata_ood_path)

In [130]:
adata_train_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/combosciplex/adata_train_{split}.h5ad"
adata_test_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/combosciplex/adata_test_{split}.h5ad"
adata_ood_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/combosciplex/adata_ood_{split}.h5ad"
adata_train = sc.read_h5ad(adata_train_path)
adata_test = sc.read_h5ad(adata_test_path)
adata_ood = sc.read_h5ad(adata_ood_path)


In [131]:
adata_test.uns.keys()

dict_keys(['Drug1_colors', 'Drug2_colors', 'Well_colors', 'condition_colors', 'dendrogram_leiden', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'ood_1_colors', 'ood_2_colors', 'ood_3_colors', 'ood_4_colors', 'pathway1_colors', 'pathway2_colors', 'pathway_colors', 'pca', 'rank_genes_groups', 'rank_genes_groups_cov_all', 'split_colors', 'umap'])

In [132]:
def prepare_data(adata_train, adata_test, adata_ood):
    
    adata_tmp =  adata_train[adata_train.obs["Drug1"].drop_duplicates().index]
    ecfp_dict = {drug: adata_tmp[adata_tmp.obs["Drug1"]==drug].obsm["ecfp_drug_1"] for drug in adata_tmp.obs["Drug1"]}

    adata_tmp =  adata_train[adata_train.obs["Drug2"].drop_duplicates().index]
    ecfp_dict.update({drug: adata_tmp[adata_tmp.obs["Drug2"]==drug].obsm["ecfp_drug_2"] for drug in adata_tmp.obs["Drug2"]})

    adata_tmp =  adata_ood[adata_ood.obs["Drug1"].drop_duplicates().index]
    ecfp_dict.update({drug: adata_tmp[adata_tmp.obs["Drug1"]==drug].obsm["ecfp_drug_1"] for drug in adata_tmp.obs["Drug1"]})

    adata_tmp =  adata_ood[adata_ood.obs["Drug2"].drop_duplicates().index]
    ecfp_dict.update({drug: adata_tmp[adata_tmp.obs["Drug2"]==drug].obsm["ecfp_drug_2"] for drug in adata_tmp.obs["Drug2"]})

        
    adata_train.uns['ecfp_rep'] = ecfp_dict
    adata_test.uns['ecfp_rep'] = ecfp_dict
    adata_ood.uns['ecfp_rep'] = ecfp_dict
    return adata_train, adata_test, adata_ood

In [141]:
adata_train, adata_test, adata_ood = prepare_data(adata_train, adata_test, adata_ood)


In [134]:
adata_train.uns["ecfp_rep"]

{'control': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Cediranib': ArrayView([[0., 0., 0., ..., 0., 0., 1.]]),
 'Panobinostat': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Dacinostat': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Givinostat': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Alvespimycin': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'SRT2104': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'PCI-34051': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'SRT3025': ArrayView([[0., 0., 1., ..., 0., 0., 0.]]),
 'Sorafenib': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Danusertib': ArrayView([[0., 1., 0., ..., 0., 0., 0.]]),
 'Tanespimycin': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Carmofur': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'SRT1720': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Crizotinib': ArrayView([[0., 1., 0., ..., 0., 0., 0.]]),
 'Pirarubicin': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Curcumin': ArrayView([[0., 0., 0., ..., 0., 0., 0.]]),
 'Dasatin

In [137]:
adata_train.obs["condition"]

Cell
A01_A02_RT_BC_10_Lig_BC_18     control+Panobinostat
A01_A02_RT_BC_10_Lig_BC_25     control+Panobinostat
A01_A02_RT_BC_10_Lig_BC_48     control+Panobinostat
A01_A02_RT_BC_10_Lig_BC_51     control+Panobinostat
A01_A02_RT_BC_10_Lig_BC_72     control+Panobinostat
                                      ...          
H12_A02_RT_BC_90_Lig_BC_73    Givinostat+Crizotinib
H12_A02_RT_BC_90_Lig_BC_83    Givinostat+Crizotinib
H12_A02_RT_BC_9_Lig_BC_41      Dacinostat+Dasatinib
H12_A02_RT_BC_9_Lig_BC_69      Dacinostat+Dasatinib
H12_A02_RT_BC_9_Lig_BC_83      Dacinostat+Dasatinib
Name: condition, Length: 44698, dtype: category
Categories (25, object): ['Alvespimycin+Pirarubicin', 'Cediranib+PCI-34051', 'Dacinostat+Danusertib', 'Dacinostat+Dasatinib', ..., 'control+Dacinostat', 'control+Givinostat', 'control+Panobinostat', 'control+SRT2104']

In [140]:
adata_train.uns.keys()



dict_keys(['Drug1_colors', 'Drug2_colors', 'Well_colors', 'condition_colors', 'dendrogram_leiden', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'ood_1_colors', 'ood_2_colors', 'ood_3_colors', 'ood_4_colors', 'pathway1_colors', 'pathway2_colors', 'pathway_colors', 'pca', 'rank_genes_groups', 'rank_genes_groups_cov_all', 'split_colors', 'umap', 'ecfp_rep'])

In [146]:

def get_train_embeddings(adata_train: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_train.obs.drop_duplicates()
    emb_vectors = {}
    for _,row in conds.iterrows():
        d_1 = row["Drug1"]
        d_2  = row["Drug2"]
        if d_1=="control" and d_2=="control":
            continue
        elif d_1 != "control" and d_2!= "control":
            emb_vectors[(d_1, d_2)] = (embeddings[d_1] + embeddings[d_2])/2.0
        elif d_1 != "control" and d_2=="control":
            emb_vectors[(d_1, d_2)] = embeddings[d_1]
        elif d_1=="control" and d_2!="control":
            emb_vectors[(d_1, d_2)] = embeddings[d_2]
    return emb_vectors
        

def find_closest_emb(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb


embeddings = adata_train.uns["ecfp_rep"]
train_embeddings = get_train_embeddings(adata_train, embeddings)
conds = adata_ood.obs.drop_duplicates(subset=["condition"])
preds = {}

for _,row in conds.iterrows():    
    if row["condition"] == "control":
        continue
    elif row["Drug2"] =="control":
        emb_0 = row["Drug1"]
        closest_emb = find_closest_emb(embeddings[emb_0], train_embeddings)
    elif row["Drug1"] =="control":
        emb_0 = row["Drug2"]
        closest_emb = find_closest_emb(embeddings[emb_0], train_embeddings)
    else:
        emb_0 = (embeddings[row["Drug1"]] + embeddings[row["Drug2"]])/2.0
        closest_emb = find_closest_emb(emb_0, train_embeddings)
    d1, d2 = row["Drug1"], row["Drug2"]
    print(closest_emb)
    preds[row["condition"]] = adata_train[adata_train.obs["condition"]==f"{closest_emb[0]}+{closest_emb[1]}"].X.toarray()
        

('Givinostat', 'SRT2104')
('Dacinostat', 'PCI-34051')
('Dacinostat', 'Dasatinib')
('Panobinostat', 'SRT1720')
('Dacinostat', 'Dasatinib')
('Panobinostat', 'SRT3025')
('Dacinostat', 'Dasatinib')
('Givinostat', 'Carmofur')


In [167]:
adata_train.obs["Drug1"].unique()

['control', 'Cediranib', 'Panobinostat', 'Dacinostat', 'Givinostat', 'Alvespimycin', 'SRT2104']
Categories (7, object): ['Alvespimycin', 'Cediranib', 'Dacinostat', 'Givinostat', 'Panobinostat', 'SRT2104', 'control']

In [164]:
conds["condition"]

Cell
A01_A02_RT_BC_16_Lig_BC_2         Givinostat+SRT1720
A01_A02_RT_BC_1_Lig_BC_3      Panobinostat+PCI-34051
A01_A02_RT_BC_40_Lig_BC_18    Panobinostat+Dasatinib
A01_A02_RT_BC_49_Lig_BC_10      Panobinostat+SRT2104
A01_A02_RT_BC_76_Lig_BC_1       Givinostat+Dasatinib
A01_A02_RT_BC_91_Lig_BC_13         SRT3025+Cediranib
A01_A02_RT_BC_94_Lig_BC_16         control+Dasatinib
A01_A02_RT_BC_59_Lig_BC_11                   control
Name: condition, dtype: category
Categories (8, object): ['Givinostat+Dasatinib', 'Givinostat+SRT1720', 'Panobinostat+Dasatinib', 'Panobinostat+PCI-34051', 'Panobinostat+SRT2104', 'SRT3025+Cediranib', 'control', 'control+Dasatinib']

In [150]:
all_data = []
conditions = []

for condition, array in preds.items():
    
    all_data.append(array)
    conditions.extend([condition] * array.shape[0])

# Stack all data vertically to create a single array
all_data_array = np.vstack(all_data)

# Create a DataFrame for the .obs attribute
obs_data = pd.DataFrame({
    'condition': conditions
})

# Create the Anndata object
adata_pred_ood = ad.AnnData(X=all_data_array, obs=obs_data)

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [151]:
adata_pred_ood

AnnData object with n_obs × n_vars = 14951 × 2000
    obs: 'condition'

In [156]:

adata_ood.obs.head()

,sample,Size_Factor,n.umi,RT_well,Drug1,Drug2,Well,n_genes,n_genes_by_counts,total_counts,...,split,control,cell_type,cell_line,smiles_drug_1,smiles_drug_2,ood_1,ood_2,ood_3,ood_4
Cell,,,,,,,,,,,,,,,,,,,,,
A01_A02_RT_BC_16_Lig_BC_2,sciPlex_theis,1.298964,3487,RT_16,Givinostat,SRT1720,B4,2154,2152,3485.0,...,ood,0,A549,A549,CCN(CC)CC1=CC2=C(C=C1)C=C(C=C2)COC(=O)NC3=CC=C...,C1CN(CCN1)CC2=CSC3=NC(=CN23)C4=CC=CC=C4NC(=O)C...,Givinostat+SRT1720,not ood,not ood,not ood
A01_A02_RT_BC_16_Lig_BC_47,sciPlex_theis,0.638120,1713,RT_16,Givinostat,SRT1720,B4,1281,1281,1713.0,...,ood,0,A549,A549,CCN(CC)CC1=CC2=C(C=C1)C=C(C=C2)COC(=O)NC3=CC=C...,C1CN(CCN1)CC2=CSC3=NC(=CN23)C4=CC=CC=C4NC(=O)C...,Givinostat+SRT1720,not ood,not ood,not ood
A01_A02_RT_BC_16_Lig_BC_55,sciPlex_theis,0.636630,1709,RT_16,Givinostat,SRT1720,B4,1269,1268,1708.0,...,ood,0,A549,A549,CCN(CC)CC1=CC2=C(C=C1)C=C(C=C2)COC(=O)NC3=CC=C...,C1CN(CCN1)CC2=CSC3=NC(=CN23)C4=CC=CC=C4NC(=O)C...,Givinostat+SRT1720,not ood,not ood,not ood
A01_A02_RT_BC_16_Lig_BC_57,sciPlex_theis,3.330295,8940,RT_16,Givinostat,SRT1720,B4,4199,4198,8939.0,...,ood,0,A549,A549,CCN(CC)CC1=CC2=C(C=C1)C=C(C=C2)COC(=O)NC3=CC=C...,C1CN(CCN1)CC2=CSC3=NC(=CN23)C4=CC=CC=C4NC(=O)C...,Givinostat+SRT1720,not ood,not ood,not ood
A01_A02_RT_BC_16_Lig_BC_6,sciPlex_theis,0.847474,2275,RT_16,Givinostat,SRT1720,B4,1584,1581,2272.0,...,ood,0,A549,A549,CCN(CC)CC1=CC2=C(C=C1)C=C(C=C2)COC(=O)NC3=CC=C...,C1CN(CCN1)CC2=CSC3=NC(=CN23)C4=CC=CC=C4NC(=O)C...,Givinostat+SRT1720,not ood,not ood,not ood


In [161]:
adata_ref = ad.concat((adata_train, adata_test, adata_ood[adata_ood.obs["control"]==0]))
cfpp.centered_pca(adata_ref, n_comps=10)


cfpp.project_pca(query_adata=adata_pred_ood, ref_adata=adata_ref)
cfpp.project_pca(query_adata=adata_ood, ref_adata=adata_ref)
ood_data_target_encoded = {}
ood_data_target_decoded = {}
ood_data_target_encoded_predicted = {}
ood_data_target_decoded_predicted = {}
for cond in adata_ood.obs["condition"].cat.categories:
    if cond == "control":
        continue
    ood_data_target_encoded[cond] = adata_ood[adata_ood.obs["condition"] == cond].obsm["X_pca"]
    ood_data_target_decoded[cond] = adata_ood[adata_ood.obs["condition"] == cond].X.toarray()
    ood_data_target_decoded_predicted[cond] = adata_pred_ood[adata_pred_ood.obs["condition"] == cond].X.toarray()
    ood_data_target_encoded_predicted[cond] = adata_pred_ood[adata_pred_ood.obs["condition"] == cond].obsm["X_pca"]


ood_deg_dict = {
    k: v
    for k, v in adata_train.uns["rank_genes_groups_cov_all"].items()
    if k in ood_data_target_decoded_predicted.keys()
}


def get_mask(x, y):
    return x[:, [gene in y for gene in adata_train.var_names]]


ood_deg_target_decoded_predicted = jax.tree_util.tree_map(get_mask, ood_data_target_decoded_predicted, ood_deg_dict)
ood_deg_target_decoded = jax.tree_util.tree_map(get_mask, ood_data_target_decoded, ood_deg_dict)


ood_metrics_encoded = jax.tree_util.tree_map(
    compute_metrics_fast, ood_data_target_encoded, ood_data_target_encoded_predicted
)

ood_metrics_decoded = jax.tree_util.tree_map(
    compute_metrics_fast, ood_data_target_decoded, ood_data_target_decoded_predicted
)

ood_metrics_deg = jax.tree_util.tree_map(
    compute_metrics_fast, ood_deg_target_decoded, ood_deg_target_decoded_predicted
)



In [162]:
ood_metrics_decoded

{'Givinostat+Dasatinib': {'r_squared': 0.5691308975219727,
  'e_distance': 598.0380859375,
  'mmd': 0.0013044075},
 'Givinostat+SRT1720': {'r_squared': 0.9931246042251587,
  'e_distance': 11.29638671875,
  'mmd': 0.00088645675},
 'Panobinostat+Dasatinib': {'r_squared': 0.8163385391235352,
  'e_distance': 241.828369140625,
  'mmd': 0.001402583},
 'Panobinostat+PCI-34051': {'r_squared': 0.8025931715965271,
  'e_distance': 281.09130859375,
  'mmd': 0.00087290094},
 'Panobinostat+SRT2104': {'r_squared': 0.9862257242202759,
  'e_distance': 18.489013671875,
  'mmd': 0.0010873616},
 'SRT3025+Cediranib': {'r_squared': 0.08479386568069458,
  'e_distance': 1368.203369140625,
  'mmd': 0.0009046337},
 'control+Dasatinib': {'r_squared': 0.48955756425857544,
  'e_distance': 666.078369140625,
  'mmd': 0.0013185237}}

In [165]:
ood_metrics_deg

{'Givinostat+Dasatinib': {'r_squared': 0.19812428951263428,
  'e_distance': 127.94894409179688,
  'mmd': 0.052633684},
 'Givinostat+SRT1720': {'r_squared': 0.9872941374778748,
  'e_distance': 1.9044952392578125,
  'mmd': 0.001525525},
 'Panobinostat+Dasatinib': {'r_squared': 0.5283968448638916,
  'e_distance': 99.97364807128906,
  'mmd': 0.03727682},
 'Panobinostat+PCI-34051': {'r_squared': 0.5274641513824463,
  'e_distance': 118.96450805664062,
  'mmd': 0.045663923},
 'Panobinostat+SRT2104': {'r_squared': 0.9855310320854187,
  'e_distance': 3.829864501953125,
  'mmd': 0.0026038678},
 'SRT3025+Cediranib': {'r_squared': 0.587827742099762,
  'e_distance': 90.87171936035156,
  'mmd': 0.072434604},
 'control+Dasatinib': {'r_squared': 0.6291071772575378,
  'e_distance': 85.14678192138672,
  'mmd': 0.0652357}}

In [166]:
ood_metrics_encoded

{'Givinostat+Dasatinib': {'r_squared': -3.0664286613464355,
  'e_distance': 579.1864318847656,
  'mmd': 0.1628626},
 'Givinostat+SRT1720': {'r_squared': 0.9714319109916687,
  'e_distance': 4.5794677734375,
  'mmd': 0.002219695},
 'Panobinostat+Dasatinib': {'r_squared': 0.4808160662651062,
  'e_distance': 220.3466339111328,
  'mmd': 0.11396325},
 'Panobinostat+PCI-34051': {'r_squared': 0.4372168183326721,
  'e_distance': 267.0052947998047,
  'mmd': 0.12374411},
 'Panobinostat+SRT2104': {'r_squared': 0.9770795106887817,
  'e_distance': 9.179443359375,
  'mmd': 0.0067232684},
 'SRT3025+Cediranib': {'r_squared': -4.737272262573242,
  'e_distance': 1354.5381164550781,
  'mmd': 0.20594288},
 'control+Dasatinib': {'r_squared': -2.0054707527160645,
  'e_distance': 632.5517425537109,
  'mmd': 0.18492188}}

In [168]:
split=5
adata_train_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex/adata_train_{split}.h5ad"
adata_test_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex/adata_test_{split}.h5ad"
adata_ood_path = f"/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex/adata_ood_{split}.h5ad"

adata_train = sc.read(adata_train_path)
adata_test = sc.read(adata_test_path)
adata_ood = sc.read(adata_ood_path)

adata_ref = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/sciplex/full_adata_with_splits.h5ad")
cfpp.centered_pca(adata_ref, n_comps=20)

In [169]:
cfpp.project_pca(query_adata=adata_ood, ref_adata=adata_ref)

In [170]:
adata_train.obs.head()

,cell_type,dose,dose_character,dose_pattern,g1s_score,g2m_score,pathway,pathway_level_1,pathway_level_2,product_dose,...,pubchem_name,pubchem_ID,smiles,control,ood_1,ood_2,ood_3,ood_4,ood_5,split
index,,,,,,,,,,,,,,,,,,,,,
A01_E09_RT_BC_100_Lig_BC_306-0-0,A549,10.0,10,4,0.000000,1.980748,DNA Damage,DNA damage & DNA repair,Nucleotide analog,Raltitrexed_10,...,Raltitrexed,135400182,CC1=NC2=C(C=C(C=C2)CN(C)C3=CC=C(S3)C(=O)NC(CCC...,False,not ood,not ood,not ood,A549_Raltitrexed_10.0,not ood,train
A01_E09_RT_BC_101_Lig_BC_229-0-0,A549,10.0,10,4,1.817254,2.801225,Apoptosis,Protein folding & Protein degradation,E3 ubiquitin ligase activity,Lenalidomide (CC-5013)_10,...,Lenalidomide,216326,C1CC(=O)NC(=O)C1N2CC3=C(C2=O)C=CC=C3N,False,not ood,not ood,not ood,A549_Lenalidomide_(CC-5013)_10.0,not ood,train
A01_E09_RT_BC_102_Lig_BC_215-0-0,A549,1000.0,1000,2,0.882834,0.882834,DNA Damage,Epigenetic regulation,Histone deacetylation,Sodium Phenylbutyrate_1000,...,Sodium 4-phenylbutyrate,5258,C1=CC=C(C=C1)CCCC(=O)[O-].[Na+],False,not ood,A549_Sodium_Phenylbutyrate_1000.0,not ood,not ood,not ood,train
A01_E09_RT_BC_102_Lig_BC_262-0-0,A549,100.0,100,3,1.058848,3.180855,Neuronal Signaling,Other,Cyclooxigenase activity,Celecoxib_100,...,celecoxib,2662,CC1=CC=C(C=C1)C2=CC(=NN2C3=CC=C(C=C3)S(=O)(=O)...,False,A549_Celecoxib_100.0,not ood,not ood,not ood,not ood,train
A01_E09_RT_BC_102_Lig_BC_332-0-0,A549,1000.0,1000,2,1.742957,2.088318,Ubiquitin,Protein folding & Protein degradation,E3 ubiquitin ligase activity,Thalidomide_1000,...,thalidomide,5426,C1CC(=O)NC(=O)C1N2C(=O)C3=CC=CC=C3C2=O,False,not ood,not ood,A549_Thalidomide_1000.0,not ood,not ood,train


In [172]:
adata_train.obs["drug"].head()

index
A01_E09_RT_BC_100_Lig_BC_306-0-0               Raltitrexed
A01_E09_RT_BC_101_Lig_BC_229-0-0    Lenalidomide_(CC-5013)
A01_E09_RT_BC_102_Lig_BC_215-0-0     Sodium_Phenylbutyrate
A01_E09_RT_BC_102_Lig_BC_262-0-0                 Celecoxib
A01_E09_RT_BC_102_Lig_BC_332-0-0               Thalidomide
Name: drug, dtype: category
Categories (180, object): ['(+)-JQ1', '2-Methoxyestradiol_(2-MeOE2)', 'A-366', 'ABT-737', ..., 'XAV-939', 'YM155_(Sepantronium_Bromide)', 'ZM_447439', 'Zileuton']

In [201]:
adata_train.obs["logdose"].unique()

[1.0, 3.0, 2.0, 4.0, 0.0]
Categories (5, float64): [0.0, 1.0, 2.0, 3.0, 4.0]

In [202]:

def get_train_embeddings(adata_train: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_train.obs.drop_duplicates(subset="condition")
    emb_vectors = {}
    for _,row in conds.iterrows():
        d_1 = row["drug"]
        if d_1=="Vehicle":
            continue
        emb_vectors[d_1] = embeddings[d_1]
    return emb_vectors
        

def find_closest_emb(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb


embeddings = adata_train.uns["ecfp_dict"]
train_embeddings = get_train_embeddings(adata_train, embeddings)
conds = adata_ood.obs.drop_duplicates(subset=["condition"])
preds = {}

for _,row in conds.iterrows():    
    if row["drug"] == "Vehicle":
        continue
    else:
        emb_0 = row["drug"]
        closest_emb = find_closest_emb(embeddings[emb_0], train_embeddings)
    print(closest_emb)
    adata_tmp = adata_train[(adata_train.obs["drug"]==closest_emb) & (adata_train.obs["cell_type"]==row["cell_type"])]
    assert adata_tmp.n_obs > 0
    adata_candidate = adata_tmp[(adata_tmp.obs["dose"]==row["dose"])]
    if adata_candidate.n_obs == 0:
        adata_candidate = adata_tmp[(adata_tmp.obs["logdose"]==row["logdose"]-1)]
        if adata_candidate.n_obs == 0:
            adata_candidate = adata_tmp[(adata_tmp.obs["logdose"]==row["logdose"]+1)]
    assert adata_candidate.n_obs > 0    
    preds[row["condition"]] = adata_candidate.X.toarray()
        

Tubastatin_A_HCl
Meprednisone
PD98059
WP1066
Nintedanib_(BIBF_1120)
Panobinostat_(LBH589)
Vandetanib_(ZD6474)
PD98059
Nintedanib_(BIBF_1120)
Panobinostat_(LBH589)
Vandetanib_(ZD6474)
Lenalidomide_(CC-5013)
PD98059
Lenalidomide_(CC-5013)
PCI-34051
WP1066
WP1066
PD98059
Vandetanib_(ZD6474)
Meprednisone
PCI-34051
Panobinostat_(LBH589)
PCI-34051
Lenalidomide_(CC-5013)
Tubastatin_A_HCl
PCI-34051
Nintedanib_(BIBF_1120)
Lenalidomide_(CC-5013)
Vandetanib_(ZD6474)
WP1066
Meprednisone
Lenalidomide_(CC-5013)
Panobinostat_(LBH589)
PCI-34051
Tubastatin_A_HCl
Nintedanib_(BIBF_1120)
Panobinostat_(LBH589)
PCI-34051
Tubastatin_A_HCl
Nintedanib_(BIBF_1120)
Nintedanib_(BIBF_1120)
Panobinostat_(LBH589)
WP1066
PCI-34051
Panobinostat_(LBH589)
Vandetanib_(ZD6474)
PCI-34051
Meprednisone
WP1066
PD98059
WP1066
Meprednisone
Vandetanib_(ZD6474)
Tubastatin_A_HCl
Lenalidomide_(CC-5013)
PD98059
Nintedanib_(BIBF_1120)
Tubastatin_A_HCl
Lenalidomide_(CC-5013)
Meprednisone
PD98059
Nintedanib_(BIBF_1120)
Lenalidomide_(CC

In [203]:
all_data = []
conditions = []

for condition, array in preds.items():
    
    all_data.append(array)
    conditions.extend([condition] * array.shape[0])

# Stack all data vertically to create a single array
all_data_array = np.vstack(all_data)

# Create a DataFrame for the .obs attribute
obs_data = pd.DataFrame({
    'condition': conditions
})

# Create the Anndata object
adata_pred_ood = ad.AnnData(X=all_data_array, obs=obs_data)

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [205]:
cfpp.project_pca(query_adata=adata_pred_ood, ref_adata=adata_ref)
cfpp.project_pca(query_adata=adata_ood, ref_adata=adata_ref)
ood_data_target_encoded = {}
ood_data_target_decoded = {}
ood_data_target_encoded_predicted = {}
ood_data_target_decoded_predicted = {}
for cond in adata_ood.obs["condition"].cat.categories:
    if "Vehicle" in cond:
        continue
    ood_data_target_encoded[cond] = adata_ood[adata_ood.obs["condition"] == cond].obsm["X_pca"]
    ood_data_target_decoded[cond] = adata_ood[adata_ood.obs["condition"] == cond].X.toarray()
    ood_data_target_decoded_predicted[cond] = adata_pred_ood[adata_pred_ood.obs["condition"] == cond].X.toarray()
    ood_data_target_encoded_predicted[cond] = adata_pred_ood[adata_pred_ood.obs["condition"] == cond].obsm["X_pca"]


ood_deg_dict = {
    k: v
    for k, v in adata_train.uns["rank_genes_groups_cov_all"].items()
    if k in ood_data_target_decoded_predicted.keys()
}


def get_mask(x, y):
    return x[:, [gene in y for gene in adata_train.var_names]]


ood_deg_target_decoded_predicted = jax.tree_util.tree_map(get_mask, ood_data_target_decoded_predicted, ood_deg_dict)
ood_deg_target_decoded = jax.tree_util.tree_map(get_mask, ood_data_target_decoded, ood_deg_dict)


ood_metrics_encoded = jax.tree_util.tree_map(
    compute_metrics_fast, ood_data_target_encoded, ood_data_target_encoded_predicted
)

ood_metrics_decoded = jax.tree_util.tree_map(
    compute_metrics_fast, ood_data_target_decoded, ood_data_target_decoded_predicted
)

ood_metrics_deg = jax.tree_util.tree_map(
    compute_metrics_fast, ood_deg_target_decoded, ood_deg_target_decoded_predicted
)



In [191]:
(set(ood_data_target_encoded.keys())) - set(ood_data_target_encoded_predicted.keys()) 

set()

In [204]:
jax.tree_util.tree_map(lambda x: x.shape[0] == 0, ood_data_target_encoded_predicted)

{'A549_Alvespimycin_(17-DMAG)_HCl_10.0': False,
 'A549_Alvespimycin_(17-DMAG)_HCl_100.0': False,
 'A549_Belinostat_(PXD101)_10.0': False,
 'A549_Belinostat_(PXD101)_100.0': False,
 'A549_Belinostat_(PXD101)_1000.0': False,
 'A549_Dacinostat_(LAQ824)_10.0': False,
 'A549_Dacinostat_(LAQ824)_100.0': False,
 'A549_Dacinostat_(LAQ824)_1000.0': False,
 'A549_Flavopiridol_HCl_10.0': False,
 'A549_Flavopiridol_HCl_100.0': False,
 'A549_Flavopiridol_HCl_1000.0': False,
 'A549_Flavopiridol_HCl_10000.0': False,
 'A549_Givinostat_(ITF2357)_10.0': False,
 'A549_Givinostat_(ITF2357)_100.0': False,
 'A549_Givinostat_(ITF2357)_1000.0': False,
 'A549_Givinostat_(ITF2357)_10000.0': False,
 'A549_Hesperadin_10.0': False,
 'A549_Hesperadin_100.0': False,
 'A549_Hesperadin_1000.0': False,
 'A549_Quisinostat_(JNJ-26481585)_2HCl_10.0': False,
 'A549_Quisinostat_(JNJ-26481585)_2HCl_100.0': False,
 'A549_TAK-901_10.0': False,
 'A549_TAK-901_100.0': False,
 'A549_TAK-901_1000.0': False,
 'A549_Tanespimycin_(17

In [207]:
ood_metrics_decoded_10 = {k:v for k,v in ood_metrics_decoded.items() if "_10.0" in k}

In [209]:
np.mean([v["r_squared"] for v in ood_metrics_decoded_10.values()])

0.9167204388865718

In [210]:
ood_metrics_decoded_100 = {k:v for k,v in ood_metrics_decoded.items() if "_100.0" in k}

In [211]:
np.mean([v["r_squared"] for v in ood_metrics_decoded_100.values()])

0.8538927206626306

In [212]:
ood_metrics_decoded_1000 = {k:v for k,v in ood_metrics_decoded.items() if "_1000.0" in k}

In [213]:
np.mean([v["r_squared"] for v in ood_metrics_decoded_1000.values()])

0.8672603680973962

In [214]:
ood_metrics_decoded_10000 = {k:v for k,v in ood_metrics_decoded.items() if "_10000.0" in k}

In [216]:
np.mean([v["r_squared"] for v in ood_metrics_decoded_10000.values()])

0.6185685396194458